# 3.2.1 파이썬 리스트에서 파이토치 텐서로
텐서 인덱싱과 비교하기 위해 우선 list 인덱싱을 실제로 해보자.

In [1]:
a = [1.0, 2.0, 1.0]

In [2]:
a[0]

1.0

In [3]:
a[2] = 3.0

In [4]:
a

[1.0, 2.0, 3.0]

## 3.2.2 첫 텐서 만들어보기

In [6]:
import torch
a = torch.ones(3) # 크기가 3인 1차원 텐서를 만들고 값을 1로 채우기
a

tensor([1., 1., 1.])

In [7]:
a[1]

tensor(1.)

In [8]:
float(a[1])

1.0

In [9]:
a[2] = 2.0
a

tensor([1., 1., 2.])

torch 모듈을 임포트하고 나면 1.0이라는 값으로 채워진, 크기가 3인(1차원) 텐서를 만드는 함수를 호출

겉으로는 숫자 객체의 리스트처럼 보이지만 내부 동작은 완전히 다름

## 3.2.3 텐서의 핵심
파이토치 텐서나 넘파이 배열은 파이썬 객체가 아닌 언박싱된 C언어 숫자 타입을 포함한 연속적인 메모리가 할당

각 요소는 32비트(4바이트) float 타입

In [12]:
points = torch.zeros(6)
points[0] = 4.0
points[1] = 1.0
points[2] = 5.0
points[3] = 3.0
points[4] = 2.0
points[5] = 1.0

# 생성자에 파이썬 리스트를 넘겨도 된다
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0])
3 points

tensor([4., 1., 5., 3., 2., 1.])

In [13]:
# 첫번째 점의 좌표를 읽을 때
float(points[0]), float(points[1])

(4.0, 1.0)

In [14]:
# 좌표 대신 2차원 좌표를 바로 인덱싱(2차원 텐서 사용)
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [15]:
# 리스트의 리스트를 생성자로 넘겨주고 텐서의 차원을 살펴보기
points.shape

torch.Size([3, 2])

In [16]:
# 텐서 초기화를 위해 차원별 크기 정보를 튜플로 만들어 zeros나 ones로 넘겨줄 수도 있다.
points = torch.zeros(3, 2)
points

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

In [17]:
# 이제 두 개의 인덱스로 텐서 내 요소를 개별적으로 접근할 수 있다.
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [18]:
points[0, 1]

tensor(1.)

In [19]:
# 데이터셋의 0번째 포인트의 Y 좌표를 반환한다. 2차원 좌표를 텐서에서 얻어보자
points[0]

tensor([4., 1.])

`동일한 데이터`를 다른 형태의 `뷰`로 표현하는 또 다른 텐서를 보여줌.

효율적인 메모리를 할당하는 방식은 3.7에서 설명.

## 텐서 인덱싱
"만일 모든 포인트에서 첫 번째 값만 구해야 한다면?"

범위 인덱싱 사용

In [20]:
some_list = list(range(6))
some_list[:] # list의 모든 요소
some_list[1:4] # 1번 인덱스~3번 인덱스
some_list[1:] # 1번 인덱스~끝
some_list[:4] # 첫번째 요소~4번 인덱스
some_list[:-1] # 첫번재 요소~마지막 요소 바로 앞
some_list[1:4:2] # 1번 인덱스~4번 인덱스까지, 두 단계 씩

[1, 3]

In [21]:
points[1:] # 첫번째 이후~모든 행 암묵적 모든 열 포함
points[1:, :] # 첫번째 이후~모든 행, 명시적 모든 열 포함
points[1:, 0] # 첫번째 이후~모든 행, 첫번째 열 만 포함
points[None] # 길이가 1인 차원을 추가(unsqueeze 동일)

tensor([[[4., 1.],
         [5., 3.],
         [2., 1.]]])

## 3.4 이름이 있는 텐서
텐서는 차원이나 축이 있고, 각 차원은 픽셀 위치나 컬러 채널에 해당

텐서접근->차원의 순서를 기억해서 인덱싱(실수 방지)

In [22]:
# 이미지 데이터->흑백 변환 가정, 여러 색상별 가중치 보고 하나의 밝기 값 뽑아내기
img_t = torch.randn(3, 5, 5) # shape [channels, rows, columns]
weights = torch.tensor([0.2126, 0.7152, 0.0722])

In [23]:
# 2차원 텐서의 흑백 이미지로부터 RGB 값을 담을 3번째 채널 자원을 더하는(배치) 작업
batch_t = torch.randn(2, 3, 5, 5) # shape [batch, channels, rows, columns]

In [24]:
# RGB 채널은 -3 차원에 있는 것으로 일반화
img_gray_naive = img_t.mean(-3)
batch_gray_naive = batch_t.mean(-3)
img_gray_naive.shape, batch_gray_naive.shape

(torch.Size([5, 5]), torch.Size([2, 5, 5]))

파이토치는 동일한 차원 정보의 텐서끼리 연산, 각 차원의 길이가 1인 텐서도 가능. 혹은 길이가 1인 차원을 알아서 늘려줌 이런 방식은

"브로드캐스팅"

In [25]:
# (2,3,5,5) 차원을 가진 batch_t를 (3,1,1) 차원의 unsqueezed_weights 곱하면 (2,3,5,5) 차원(채널 정보를 가짐, 뒤에서 세번째 차원 값에 대한 합 구하기 가능
unsqueezed_weights = weights.unsqueeze(-1).unsqueeze_(-1)
img_weights = (img_t * unsqueezed_weights)
batch_weights = (batch_t * unsqueezed_weights)
img_gray_weights = img_weights.sum(-3) # 뒤에서 세번째 차원 값에 대한 합
batch_gray_weights = batch_weights.sum(-3)
batch_weights.shape, batch_t.shape, unsqueezed_weights.shape

(torch.Size([2, 3, 5, 5]), torch.Size([2, 3, 5, 5]), torch.Size([3, 1, 1]))

### `einsum` 함수
차원별로 이름을 부여하는 작은 언어, 점 세개(...)로 변수명 없이 합을 구하는 브로드캐스팅 사용.

In [26]:
img_gray_weighted_fancy = torch.einsum('...chw,c->...hw', img_t, weights)
batch_gray_weighted_fancy = torch.einsum('...chw,c->...hw', batch_t, weights)
batch_gray_weighted_fancy.shape

torch.Size([2, 5, 5])

보다시피 많은 기호가 복잡하게 사용. -> 차원에 이름을 부여하는 방식 제안

`tensor` 나 `rand` 같은 텐서 팩토리 함수에 사용할 문자열 리스트를 `names`인자로 전달 가능

In [29]:
# torch.tensor([, , ], names=[''])
weights_named = torch.tensor([0.2126, 0.7152, 0.0722], names=['channels'])
weights_named

tensor([0.2126, 0.7152, 0.0722], names=('channels',))

"텐서를 먼저 만들고 나중에 이름을 지정할래."
-> redefine_names 함수를 사용하면, 텐서 접근 시 인덱싱하듯 생략 부호 ...를 사용

In [28]:
# 변수명 = img.refine_names(..., '이름', '이름2', '이름3')
img_named = img_t.refine_names(..., 'channels', 'rows', 'columns')
batch_named = batch_t.refine_names(..., 'channels', 'rows', 'columns')
print("img named:", img_named.shape, img_named.names)
print("batch named:", batch_named.shape, batch_named.names)

img named: torch.Size([3, 5, 5]) ('channels', 'rows', 'columns')
batch named: torch.Size([2, 3, 5, 5]) (None, 'channels', 'rows', 'columns')


In [30]:
# align_as 함수는 빠진 차원을 채우고, 존재하는 차원을 올바른 순서로 바꿔준다(파이토치가 차원을 자동으로 정렬X)
weights_aligned = weights_named.align_as(img_named)
weights_aligned.shape, weights_aligned.names

(torch.Size([3, 1, 1]), ('channels', 'rows', 'columns'))

In [31]:
# sum 처럼 차원 인수를 허용하는 함수들은 이름이 붙은 차원도 받아들인다
gray_named = (img_named * weights_aligned).sum('channels')
gray_named.shape, gray_named.names

(torch.Size([5, 5]), ('rows', 'columns'))

In [32]:
# 이름이 다른 차원을 결합하려면 오류가 발생한다
gray_named = (img_named[..., :3] * weights_named).sum('channels')

RuntimeError: Error when attempting to broadcast dims ['channels', 'rows', 'columns'] and dims ['channels']: dim 'columns' and dim 'channels' are at the same position from the right but do not match.

In [33]:
# 이름이 있는 텐서를 사용하는 연산을 함수 밖에서 사용하고 싶다면, 차원 이름에 None을 넣어 이름 없는 텐서를 만든다
gray_plain = gray_named.rename(None)
gray_plain.shape, gray_plain.names

(torch.Size([5, 5]), (None, None))